# Working with Remote LLM APIs: OpenAI or Gemini

## Lab Three (Estimated Duration: ~2 hours)

This lab is **API-only**. It can use either **OpenAI** or **Google Gemini**. Some learners will have easier access to one provider than the other, so the notebook uses one common programming pattern and switches the provider at setup time.

You need one of these working configurations in the notebook environment:

- `LAB_THREE_PROVIDER=openai`, `OPENAI_API_KEY`, and `OPENAI_MODEL`; or
- `LAB_THREE_PROVIDER=gemini`, `GEMINI_API_KEY`, and `GEMINI_MODEL`.

If `LAB_THREE_PROVIDER` is not set, the notebook auto-selects OpenAI when OpenAI variables are present, otherwise Gemini when Gemini variables are present.

The main learning goal is not just "make the model answer". The goal is to understand each programming layer: checking credentials, building prompts, creating request payloads, sending HTTP requests, parsing responses, logging parameters, and validating outputs.

The notebook makes real remote API calls using small synthetic health and social examples. It does not use substitute responses. If credentials, model access, network access, request shape, or response parsing fails, the notebook raises an error. Fix the error before continuing.


## How to Use This Notebook

- Complete `api_setup.md` before opening this notebook.
- Choose either OpenAI or Gemini; you do not need both.
- Run cells from top to bottom after setting your environment variables.
- Use only the synthetic examples included here.
- Do not paste personal data, health records, interview transcripts, confidential institutional material, or real service-user data into prompts.
- If a cell errors, read the final line of the traceback, fix the setup or code, and rerun from the relevant earlier cell.

Each setup function is introduced in a small cell. Read the markdown before running the code: it explains what the function does and why it is needed.


## Setup Overview

The setup is split into small cells instead of one large preamble:

1. imports, endpoints, and provider constants;
2. synthetic teaching data;
3. provider selection and credential checking;
4. prompt-building functions;
5. provider-specific request builders;
6. HTTP calling and response parsing;
7. logging, status-code, and evaluation helpers.

The important design choice is that the exercise code later in the notebook calls provider-neutral functions such as `call_model(...)`, `build_provider_payload(...)`, and `make_log_entry(...)`. Those functions hide the small differences between OpenAI and Gemini request shapes.


### Setup 1: Imports, Endpoints, and Constants

This cell imports Python standard-library modules only. No third-party OpenAI or Google package is required.

The constants define:

- the OpenAI Responses API endpoint;
- the Gemini Interactions API endpoint;
- the timeout used for API calls;
- the allowed labels for classification tasks;
- the system instruction sent with every request.

OpenAI and Gemini use different JSON payload shapes, but both can be called with ordinary HTTPS requests. That is why this notebook uses `urllib.request`: it makes the transport layer visible for teaching.


In [ ]:
import hashlib
import json
import os
import re
import socket
import time
import urllib.error
import urllib.request
from datetime import datetime, timezone


OPENAI_RESPONSES_URL = "https://api.openai.com/v1/responses"
GEMINI_INTERACTIONS_URL = "https://generativelanguage.googleapis.com/v1beta/interactions"
API_TIMEOUT_SECONDS = 240

ALLOWED_PROVIDERS = ["openai", "gemini"]
ALLOWED_LABELS = ["access_barrier", "housing_stress", "health_need", "other"]

system_instruction = """
You are helping with a teaching exercise about LLMs for health and social science research.
Use only the information in the supplied text.
For classification tasks, return only one allowed label unless the prompt asks for another format.
"""


### Setup 2: Synthetic Teaching Data

These examples are deliberately small and synthetic. They are shaped like public comments or de-identified service notes, not real patient or service-user records.

The labels are used later for validation. The model may disagree with them; that is part of the exercise. A model answer is not evidence until it has been compared against a human label or another explicit standard.


In [ ]:
toy_notes = [
    {
        "id": "N001",
        "text": "Resident says the evening bus was cancelled and they missed a diabetes clinic appointment.",
        "human_label": "access_barrier",
    },
    {
        "id": "N002",
        "text": "Advice note reports rent arrears, threat of eviction, and difficulty reaching the housing office.",
        "human_label": "housing_stress",
    },
    {
        "id": "N003",
        "text": "Patient forum post asks for advice after medication side effects and trouble arranging a follow-up appointment.",
        "human_label": "health_need",
    },
]

applied_notes = [
    {
        "id": "A001",
        "text": "Public comment says the evening bus was cancelled and several residents missed diabetes clinic appointments.",
        "human_label": "access_barrier",
    },
    {
        "id": "A002",
        "text": "Community advice note reports rent arrears, threat of eviction, and difficulty reaching the housing office.",
        "human_label": "housing_stress",
    },
    {
        "id": "A003",
        "text": "Patient forum post asks for advice after medication side effects and trouble arranging a follow-up appointment.",
        "human_label": "health_need",
    },
    {
        "id": "A004",
        "text": "Local survey response praises new park lighting and weekend community activities.",
        "human_label": "other",
    },
    {
        "id": "A005",
        "text": "Public meeting note says older residents cannot afford taxis to outpatient appointments.",
        "human_label": "access_barrier",
    },
]

public_complaint_extract = """
Public feedback extract:
Several residents on the estate said the new appointment booking system is hard to use.
One carer said she tried to call the clinic three times and could not get through.
Another resident said online booking works for him, but his neighbour does not have internet access.
The group asked for a phone option, clearer opening hours, and written information in plain English.
"""

deidentified_service_note = """
Public service note:
Area=Riverside.
Issue=missed clinic appointment after bus route change.
Household also mentions rent arrears.
Requested support: transport advice and benefits referral.
No immediate safeguarding concern is stated.
"""

sensitive_training_example = """
Maria Jones called from SW1A 1AA on 12/06/2026.
Phone 07123 456789.
She missed a clinic appointment after her bus route was cancelled.
She wants advice about transport options.
"""


### Setup 3: Provider Selection and Credential Check

`require_api_setup()` decides which remote provider this notebook will use.

There are two valid ways to choose:

- set `LAB_THREE_PROVIDER` to `openai` or `gemini`; or
- leave `LAB_THREE_PROVIDER` unset and let the notebook auto-detect from available environment variables.

Provider-specific variables:

| Provider | Required key | Required model |
|---|---|---|
| OpenAI | `OPENAI_API_KEY` | `OPENAI_MODEL` |
| Gemini | `GEMINI_API_KEY` | `GEMINI_MODEL` |

The function returns a short key preview so learners can see that a key exists without printing the full secret.


In [ ]:
def key_preview(key):
    if len(key) <= 11:
        return "[set but too short to preview safely]"
    return key[:7] + "..." + key[-4:]


def detect_provider():
    requested = (os.getenv("LAB_THREE_PROVIDER") or "").strip().lower()
    if requested:
        if requested not in ALLOWED_PROVIDERS:
            raise EnvironmentError(
                "LAB_THREE_PROVIDER must be either 'openai' or 'gemini'. "
                f"Current value: {requested!r}"
            )
        return requested

    has_openai = bool(os.getenv("OPENAI_API_KEY") and os.getenv("OPENAI_MODEL"))
    has_gemini = bool(os.getenv("GEMINI_API_KEY") and os.getenv("GEMINI_MODEL"))

    if has_openai:
        return "openai"
    if has_gemini:
        return "gemini"

    raise EnvironmentError(
        "No usable Lab Three API setup found. Set either OPENAI_API_KEY + OPENAI_MODEL "
        "or GEMINI_API_KEY + GEMINI_MODEL. See api_setup.md."
    )


def require_api_setup():
    provider = detect_provider()
    if provider == "openai":
        key_name = "OPENAI_API_KEY"
        model_name = "OPENAI_MODEL"
    else:
        key_name = "GEMINI_API_KEY"
        model_name = "GEMINI_MODEL"

    missing = [name for name in [key_name, model_name] if not os.getenv(name)]
    if missing:
        raise EnvironmentError(
            "Missing required environment variable(s) for "
            + provider
            + ": "
            + ", ".join(missing)
            + ". Complete api_setup.md, restart the notebook kernel, and rerun this cell."
        )

    key = os.environ[key_name]
    model = os.environ[model_name]
    return {
        "provider": provider,
        "key_name": key_name,
        "model_name": model_name,
        "has_key": True,
        "key_preview": key_preview(key),
        "model": model,
    }


### Setup 4: Prompt Builders

These functions turn ordinary Python strings into task-specific prompts.

Why use functions here?

- The same wording can be reused across providers.
- The task instructions stay consistent.
- Provider switching happens in request-building code, not in prompt-building code.
- The notebook is easier to debug because prompt construction is separated from API calling.

`redact_for_api()` is a teaching example. It catches obvious identifiers in the synthetic note, but it is not a complete de-identification method.


In [ ]:
def make_label_prompt(text):
    return f"""
Classify the text as exactly one of:
access_barrier, housing_stress, health_need, other.

Return only the exact label.

Text:
{text}
""".strip()


def make_summary_prompt(text):
    return f"""
Summarise the public feedback extract in two short bullet points.
Then add one sentence beginning 'Uncertainty:' that states what we do not know.

Text:
{text}
""".strip()


def make_extraction_prompt(text):
    return f"""
Return valid JSON only with these keys:
area, main_issue, secondary_issue, requested_support, risk_level.

Use only the information in the note.

Note:
{text}
""".strip()


def redact_for_api(text):
    redacted = re.sub(r"\b[A-Z][a-z]+ [A-Z][a-z]+\b", "[NAME]", text)
    redacted = re.sub(r"\b0\d{4}\s?\d{6}\b", "[PHONE]", redacted)
    redacted = re.sub(r"\b[A-Z]{1,2}\d[A-Z\d]?\s?\d[A-Z]{2}\b", "[POSTCODE]", redacted)
    redacted = re.sub(r"\b\d{1,2}/\d{1,2}/\d{2,4}\b", "[DATE]", redacted)
    return redacted


### Setup 5: Provider-Specific Request Builders

The same prompt needs different JSON depending on the provider.

OpenAI Responses API request shape in this lab:

- URL: `https://api.openai.com/v1/responses`
- secret in the `Authorization: Bearer ...` header
- model in the `model` field
- prompt material in the `input` field

Gemini Interactions API request shape in this lab:

- URL: `https://generativelanguage.googleapis.com/v1beta/interactions`
- secret in the `x-goog-api-key` header
- model in the `model` field
- prompt material in the `input` field
- generation parameters inside `generation_config`

`build_provider_payload()` and `build_provider_request()` hide those differences so the exercise cells can stay provider-neutral.


In [ ]:
def build_provider_payload(prompt, temperature=None, top_p=None, max_output_tokens=80):
    setup = require_api_setup()
    provider = setup["provider"]

    if provider == "openai":
        payload = {
            "model": setup["model"],
            "input": [
                {"role": "system", "content": system_instruction.strip()},
                {"role": "user", "content": prompt},
            ],
            "max_output_tokens": max_output_tokens,
        }
        if temperature is not None:
            payload["temperature"] = temperature
        if top_p is not None:
            payload["top_p"] = top_p
        return payload

    generation_config = {"max_output_tokens": max_output_tokens}
    if temperature is not None:
        generation_config["temperature"] = temperature
    if top_p is not None:
        generation_config["top_p"] = top_p

    return {
        "model": setup["model"],
        "system_instruction": system_instruction.strip(),
        "input": prompt,
        "generation_config": generation_config,
    }


def build_provider_request(payload):
    setup = require_api_setup()
    provider = setup["provider"]

    if provider == "openai":
        key = os.environ["OPENAI_API_KEY"]
        return {
            "provider": provider,
            "url": OPENAI_RESPONSES_URL,
            "headers": {
                "Authorization": f"Bearer {key}",
                "Content-Type": "application/json",
            },
            "payload": payload,
        }

    key = os.environ["GEMINI_API_KEY"]
    return {
        "provider": provider,
        "url": GEMINI_INTERACTIONS_URL,
        "headers": {
            "x-goog-api-key": key,
            "Content-Type": "application/json",
        },
        "payload": payload,
    }


def redact_provider_request(request_spec):
    redacted_headers = dict(request_spec["headers"])
    if "Authorization" in redacted_headers:
        redacted_headers["Authorization"] = "Bearer [REDACTED]"
    if "x-goog-api-key" in redacted_headers:
        redacted_headers["x-goog-api-key"] = "[REDACTED]"

    return {
        "provider": request_spec["provider"],
        "url": request_spec["url"],
        "headers": redacted_headers,
        "payload": request_spec["payload"],
    }


### Setup 6: Calling the Provider and Parsing Text

This cell contains the actual HTTP machinery.

`post_json()` sends a JSON request and raises an error for HTTP, network, or timeout failures. A timeout is still a real failure; the notebook does not substitute an answer.

OpenAI and Gemini return different response shapes. The provider-specific parsers convert those shapes into one common result object with:

- `provider`;
- `model`;
- `status`;
- `elapsed_seconds`;
- `payload`;
- `json`;
- `text`.

Later exercise cells use that common object, so the learning workflow is the same whichever provider the student uses.


In [ ]:
def provider_label():
    return require_api_setup()["provider"].upper()


def provider_model():
    return require_api_setup()["model"]


def post_json(url, headers, payload, provider, timeout=None):
    if timeout is None:
        timeout = API_TIMEOUT_SECONDS

    data = json.dumps(payload).encode("utf-8")
    request = urllib.request.Request(url, data=data, headers=headers, method="POST")
    try:
        start = time.time()
        with urllib.request.urlopen(request, timeout=timeout) as response:
            raw_body = response.read().decode("utf-8")
        elapsed = time.time() - start
        return {
            "status": response.status,
            "elapsed_seconds": elapsed,
            "json": json.loads(raw_body),
        }
    except urllib.error.HTTPError as exc:
        body = exc.read().decode("utf-8", errors="replace")
        raise RuntimeError(f"{provider.upper()} HTTP error {exc.code}: {body}") from exc
    except urllib.error.URLError as exc:
        reason = getattr(exc, "reason", exc)
        if isinstance(reason, (TimeoutError, socket.timeout)):
            raise TimeoutError(
                f"{provider.upper()} request timed out after {timeout} seconds before a complete response was received. "
                "Rerun the cell, check the network, reduce max_output_tokens/batch size, "
                "or increase API_TIMEOUT_SECONDS in Setup 1."
            ) from exc
        raise ConnectionError(f"{provider.upper()} request failed before receiving a response: {reason}") from exc
    except (TimeoutError, socket.timeout) as exc:
        raise TimeoutError(
            f"{provider.upper()} request timed out after {timeout} seconds while waiting for the response body. "
            "Rerun the cell, check the network, reduce max_output_tokens/batch size, "
            "or increase API_TIMEOUT_SECONDS in Setup 1."
        ) from exc


def extract_text_from_openai_response(response_json):
    if response_json.get("output_text"):
        return response_json["output_text"].strip()

    chunks = []
    for item in response_json.get("output", []):
        for content in item.get("content", []):
            if isinstance(content, dict) and content.get("text"):
                chunks.append(content["text"])

    text = "\n".join(chunk.strip() for chunk in chunks if chunk.strip())
    if not text:
        raise ValueError(f"Could not find OpenAI response text. Top-level keys: {list(response_json.keys())}")
    return text


def extract_text_from_gemini_response(response_json):
    if response_json.get("output_text"):
        return response_json["output_text"].strip()

    chunks = []
    for step in response_json.get("steps", []):
        for content in step.get("content", []):
            if isinstance(content, dict):
                if content.get("text"):
                    chunks.append(content["text"])
                elif content.get("type") == "text" and content.get("value"):
                    chunks.append(content["value"])

    for candidate in response_json.get("candidates", []):
        content = candidate.get("content", {})
        for part in content.get("parts", []):
            if isinstance(part, dict) and part.get("text"):
                chunks.append(part["text"])

    text = "\n".join(chunk.strip() for chunk in chunks if chunk.strip())
    if not text:
        raise ValueError(f"Could not find Gemini response text. Top-level keys: {list(response_json.keys())}")
    return text


def extract_text_from_provider_response(provider, response_json):
    if provider == "openai":
        return extract_text_from_openai_response(response_json)
    if provider == "gemini":
        return extract_text_from_gemini_response(response_json)
    raise ValueError(f"Unknown provider: {provider}")


def call_model(prompt, temperature=None, top_p=None, max_output_tokens=80):
    setup = require_api_setup()
    payload = build_provider_payload(
        prompt=prompt,
        temperature=temperature,
        top_p=top_p,
        max_output_tokens=max_output_tokens,
    )
    request_spec = build_provider_request(payload)
    result = post_json(
        url=request_spec["url"],
        headers=request_spec["headers"],
        payload=request_spec["payload"],
        provider=setup["provider"],
    )
    text = extract_text_from_provider_response(setup["provider"], result["json"])
    return {
        "provider": setup["provider"],
        "model": setup["model"],
        "status": result["status"],
        "elapsed_seconds": result["elapsed_seconds"],
        "payload": payload,
        "json": result["json"],
        "text": text,
    }


def normalise_label(output_text):
    text = (output_text or "").strip().lower()
    text = re.sub(r"^[`'\"\s]+|[`'\"\s]+$", "", text)
    text = re.sub(r"[^a-z_ -].*$", "", text).strip()
    canonical = re.sub(r"[\s-]+", "_", text)
    canonical = re.sub(r"_+", "_", canonical).strip("_")
    compact = canonical.replace("_", "")

    label_aliases = {
        "accessbarrier": "access_barrier",
        "access_barriers": "access_barrier",
        "access_barrier": "access_barrier",
        "housingstress": "housing_stress",
        "housing_stresses": "housing_stress",
        "housing_stress": "housing_stress",
        "healthneed": "health_need",
        "health_needs": "health_need",
        "health_need": "health_need",
        "other": "other",
    }

    if canonical in ALLOWED_LABELS:
        return canonical
    if canonical in label_aliases:
        return label_aliases[canonical]
    if compact in label_aliases:
        return label_aliases[compact]
    for label in ALLOWED_LABELS:
        if canonical.startswith(label) or compact.startswith(label.replace("_", "")):
            return label
    return "invalid"


### Setup 7: Logging, Status-Code, and Evaluation Helpers

These helpers support reproducibility and debugging.

`prompt_hash()` stores a short fingerprint of a prompt without printing the whole prompt.

`make_log_entry()` records the provider, model, parameters, status, and timing for a real API call.

`explain_api_status()` gives beginner-friendly interpretations of common HTTP status codes. The codes are not provider-specific; the likely fix can still differ by account, quota, or model access.


In [ ]:
def prompt_hash(prompt):
    return hashlib.sha256(prompt.encode("utf-8")).hexdigest()[:12]


def make_log_entry(task, prompt, result, temperature=None, top_p=None, max_output_tokens=80):
    return {
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "task": task,
        "provider": result["provider"],
        "model": result["model"],
        "prompt_hash": prompt_hash(prompt),
        "temperature": temperature,
        "top_p": top_p,
        "max_output_tokens": max_output_tokens,
        "status": result["status"],
        "elapsed_seconds": result["elapsed_seconds"],
    }


def explain_api_status(status_code):
    explanations = {
        200: "Success: parse, validate, and log the response.",
        400: "Bad request: inspect endpoint, payload shape, model name, or parameters.",
        401: "Authentication problem: check key, billing, or project permissions.",
        403: "Permission problem: check key, billing, region, quota, or model access.",
        404: "Not found: check endpoint URL or model name.",
        413: "Request too large: reduce prompt, context, files, or batch size.",
        429: "Rate limited or quota-limited: wait, reduce request rate, or check quota.",
        500: "Provider-side problem: retry later with logging.",
        503: "Provider unavailable or overloaded: retry later with logging.",
    }
    return explanations.get(status_code, "Unexpected status: inspect the response body and request log.")


## Part A: Provider Setup and Prompt (Approx. 20 minutes)

In this part, you check that the notebook can see one valid provider setup and build the first prompt.

Keep the task small. A short classification prompt is easier to debug than a long, complex request. The same prompt will later be sent to whichever provider you configured.


### Question 1: Check Provider Setup

Run the setup check. It should:

- identify the selected provider;
- confirm that a key exists;
- print only a short preview of the key;
- print the model name.

If this cell errors, the notebook cannot see the required environment variables for your provider. Return to `api_setup.md`, restart the kernel, and rerun the setup cells.


In [ ]:
# Answer 1
setup_status = require_api_setup()
print("Selected provider:", setup_status["provider"])
print("Required key variable:", setup_status["key_name"])
print("Key available:", setup_status["has_key"])
print("Key preview:", setup_status["key_preview"])
print("Model variable:", setup_status["model_name"])
print("Model:", setup_status["model"])


### Question 2: Build a Toy Prompt

Start with one synthetic note. Your aim is to build a prompt that is easy to inspect before sending.

Programming steps:

- choose the first note;
- pass the note text into `make_label_prompt(...)`;
- print the prompt and check the allowed labels are clear.


In [ ]:
# Answer 2
selected_note = toy_notes[0]
prompt = make_label_prompt(selected_note["text"])

print("Selected note:", selected_note["id"])
print(prompt)


## Part B: Build and Inspect One Provider Request (Approx. 25 minutes)

Do not send a request until you understand what is in it.

In this part, you build the payload, wrap it in a request spec, and inspect a redacted version that hides the API key. The request spec will look different for OpenAI and Gemini, but the conceptual parts are the same: URL, headers, model, prompt, and generation settings.


### Question 3: Build the Provider Payload

The payload is the JSON body sent to the selected provider.

Programming steps:

- call `build_provider_payload(...)`;
- keep `max_output_tokens` small;
- print the payload with indentation;
- check the model, system instruction, user prompt, and output limit.


In [ ]:
# Answer 3
payload = build_provider_payload(
    prompt=prompt,
    temperature=None,
    top_p=None,
    max_output_tokens=30,
)

print(json.dumps(payload, indent=2))


### Question 4: Build a Redacted Request Spec

The request spec adds URL and headers. Headers contain the secret API key, so never print the raw request spec.

Programming steps:

- call `build_provider_request(...)`;
- pass that into `redact_provider_request(...)`;
- print only the redacted request.


In [ ]:
# Answer 4
request_spec = build_provider_request(payload)
redacted_request = redact_provider_request(request_spec)

print(json.dumps(redacted_request, indent=2))


## Part C: Make, Parse, and Compare Provider Calls (Approx. 35 minutes)

These cells make real API calls. They are intentionally small: one prompt, short output, synthetic text.

If a call fails, use the traceback and the status-code guidance later in the lab to identify the problem. Common causes include missing key, wrong model name, quota limits, unsupported parameters, and network timeouts.


### Question 5: Make One API Call

This is the first live remote model call.

Programming steps:

- call `call_model(...)`;
- store the result in `api_result`;
- print one clear confirmation line beginning `REMOTE API CALLED: YES`;
- print provider, status, timing, and response text.


In [ ]:
# Answer 5
api_result = call_model(
    prompt=prompt,
    temperature=None,
    top_p=None,
    max_output_tokens=30,
)

print(f"REMOTE API CALLED: YES | provider={api_result['provider']} | task=single_note_classification | model={api_result['model']} | status={api_result['status']} | elapsed_seconds={api_result['elapsed_seconds']:.2f}")
print("Provider:", api_result["provider"])
print("Status:", api_result["status"])
print("Elapsed seconds:", round(api_result["elapsed_seconds"], 2))
print("Text:", api_result["text"])


### Question 6: Parse Response Text

The parser should work on the real response from Question 5.

Programming steps:

- pass the provider name and raw JSON into `extract_text_from_provider_response(...)`;
- normalise the text into a label;
- print both values.


In [ ]:
# Answer 6
parsed_text = extract_text_from_provider_response(api_result["provider"], api_result["json"])
parsed_label = normalise_label(parsed_text)

print("Parsed text:", parsed_text)
print("Normalised label:", parsed_label)


### Question 7: Compare Two Parameter Settings

Parameters are part of the method. Even when the prompt is the same, different parameter settings can change output.

OpenAI and Gemini both support common sampling settings such as temperature, but not every model supports every parameter in the same way. If a parameter causes an error for your selected provider/model, remove that setting and document the change.

Programming steps:

- define a small list of parameter settings;
- loop over the settings;
- call the selected provider once per setting;
- print a confirmation line for each real API call;
- store the output and normalised label.


In [ ]:
# Answer 7
parameter_tests = [
    {"name": "default_sampling", "temperature": None, "top_p": None, "max_output_tokens": 30},
    {"name": "low_temperature", "temperature": 0.0, "top_p": 1.0, "max_output_tokens": 30},
]

parameter_results = []

for setting in parameter_tests:
    result = call_model(
        prompt=prompt,
        temperature=setting["temperature"],
        top_p=setting["top_p"],
        max_output_tokens=setting["max_output_tokens"],
    )
    print(f"REMOTE API CALLED: YES | provider={result['provider']} | task=parameter_comparison | setting={setting['name']} | model={result['model']} | status={result['status']} | elapsed_seconds={result['elapsed_seconds']:.2f}")
    parameter_results.append(
        {
            "name": setting["name"],
            "provider": result["provider"],
            "model": result["model"],
            "temperature": setting["temperature"],
            "top_p": setting["top_p"],
            "status": result["status"],
            "text": result["text"],
            "normalised_label": normalise_label(result["text"]),
        }
    )

print(json.dumps(parameter_results, indent=2))


## Part D: Logging, Failure Handling, and Validation (Approx. 25 minutes)

Real API work needs records. A useful log does not need to contain the whole prompt or API key, but it should record enough to reproduce the call.

Provider choice is part of the method. Record both the provider and model, because `gemini-...` and `gpt-...` names do not mean the same thing and may change over time.


### Question 8: Create a Reproducibility Log Entry

Create a compact log entry for the first real API call.

Programming steps:

- call `make_log_entry(...)`;
- pass the same parameters used in Question 5;
- print the log entry as JSON.


In [ ]:
# Answer 8
log_entry = make_log_entry(
    task="single_note_classification",
    prompt=prompt,
    result=api_result,
    temperature=None,
    top_p=None,
    max_output_tokens=30,
)

print(json.dumps(log_entry, indent=2))


### Question 9: Explain API Failure Codes

When a real call fails, the status code helps identify the next fix.

This question does not call the API. It prints a reference table for common status codes. The exact wording of an error body differs by provider, so inspect both the status code and the provider's returned message.


In [ ]:
# Answer 9
for status_code in [200, 400, 401, 403, 404, 413, 429, 500, 503]:
    print(status_code, "->", explain_api_status(status_code))


### Question 10: Validate Against the Human Label

The model output is not evidence until it is checked.

Programming steps:

- compare the model label with the human label;
- store the comparison in a dictionary;
- print the result.


In [ ]:
# Answer 10
validation_result = {
    "note_id": selected_note["id"],
    "provider": api_result["provider"],
    "model": api_result["model"],
    "human_label": selected_note["human_label"],
    "model_text": api_result["text"],
    "model_label": normalise_label(api_result["text"]),
}
validation_result["match"] = validation_result["model_label"] == validation_result["human_label"]

print(json.dumps(validation_result, indent=2))


## Part E: Applied Health and Social Workflows (Approx. 45 minutes)

Now apply the same real API pattern to four small tasks:

- batch classification;
- summarisation;
- structured extraction;
- redaction before sending text.

The examples remain synthetic and small, but the programming pattern is the same pattern used in larger workflows. Provider switching should not change the workflow: build prompt, call model, parse response, validate or inspect.


### Programming Pattern for the Applied Tasks

Each applied task follows the same sequence:

1. Put text in a variable.
2. Build a clear prompt.
3. Call the selected remote model provider.
4. Parse the response.
5. Validate, inspect, or log the output.

When debugging, reduce the workflow to one row, one prompt, and one printed response before scaling back up.


### Question 11: Batch Classify Applied Notes

Batch work means repeating the same operation over several examples.

Programming steps:

- loop over `applied_notes`;
- build one label prompt per note;
- call the selected provider once per note;
- normalise the output;
- print a confirmation line for each real API call;
- compare with the human label.


In [ ]:
# Answer 11
batch_rows = []

for note in applied_notes:
    print(f"REMOTE API CALL STARTING: provider={provider_label()} | task=batch_classification | note_id={note['id']}")
    note_prompt = make_label_prompt(note["text"])
    result = call_model(
        prompt=note_prompt,
        temperature=None,
        top_p=None,
        max_output_tokens=20,
    )
    print(f"REMOTE API CALLED: YES | provider={result['provider']} | task=batch_classification | note_id={note['id']} | model={result['model']} | status={result['status']} | elapsed_seconds={result['elapsed_seconds']:.2f}")
    model_label = normalise_label(result["text"])
    batch_rows.append(
        {
            "id": note["id"],
            "provider": result["provider"],
            "model": result["model"],
            "human_label": note["human_label"],
            "model_text": result["text"],
            "model_label": model_label,
            "match": model_label == note["human_label"],
            "status": result["status"],
            "elapsed_seconds": result["elapsed_seconds"],
        }
    )

for row in batch_rows:
    print(row)


### Question 12: Evaluate the Batch Classifier

A classifier should be evaluated as a classifier, not just inspected row by row.

Use the `batch_rows` object from Question 11 to calculate:

- overall accuracy;
- how many outputs were outside the allowed label set;
- how many examples were tested per human label;
- per-label accuracy;
- which rows disagreed with the human label.

This is still a tiny synthetic evaluation set, so the numbers are not research evidence. The point is to practise the evaluation structure.


In [ ]:
# Answer 12
total_rows = len(batch_rows)
correct_rows = 0
invalid_rows = 0
per_label_counts = {}
per_label_correct = {}
disagreement_rows = []

for row in batch_rows:
    human_label = row["human_label"]
    per_label_counts[human_label] = per_label_counts.get(human_label, 0) + 1
    per_label_correct.setdefault(human_label, 0)

    if row["match"]:
        correct_rows = correct_rows + 1
        per_label_correct[human_label] = per_label_correct[human_label] + 1
    else:
        disagreement_rows.append(row)

    if row["model_label"] not in ALLOWED_LABELS:
        invalid_rows = invalid_rows + 1

per_label_accuracy = {}
for label, count in per_label_counts.items():
    per_label_accuracy[label] = per_label_correct[label] / count

accuracy = correct_rows / total_rows

classifier_evaluation = {
    "n": total_rows,
    "provider": batch_rows[0]["provider"] if batch_rows else None,
    "model": batch_rows[0]["model"] if batch_rows else None,
    "accuracy": accuracy,
    "invalid_outputs": invalid_rows,
    "per_label_counts": per_label_counts,
    "per_label_accuracy": per_label_accuracy,
    "disagreements": disagreement_rows,
}

print(f"CLASSIFIER ACCURACY ACROSS ALL BATCH EXAMPLES: {correct_rows}/{total_rows} = {accuracy:.3f}")
print(f"CLASSIFIER INVALID OUTPUTS ACROSS ALL BATCH EXAMPLES: {invalid_rows}/{total_rows}")
print(json.dumps(classifier_evaluation, indent=2))


### Question 13: Summarise a Public Complaint Extract

Summarisation is different from classification. The output is prose, so validation requires reading for accuracy and uncertainty.

Programming steps:

- build a summary prompt;
- call the selected provider with a slightly larger output limit;
- print the summary.


In [ ]:
# Answer 13
summary_prompt = make_summary_prompt(public_complaint_extract)
summary_result = call_model(
    prompt=summary_prompt,
    temperature=None,
    top_p=None,
    max_output_tokens=160,
)

print(f"REMOTE API CALLED: YES | provider={summary_result['provider']} | task=summarisation | model={summary_result['model']} | status={summary_result['status']} | elapsed_seconds={summary_result['elapsed_seconds']:.2f}")
summary_text = summary_result["text"]
print(summary_text)


### Question 14: Extract Structured Fields

Structured extraction is useful because the output can be parsed into a Python dictionary.

Programming steps:

- ask for valid JSON only;
- call the selected provider;
- parse the returned text with `json.loads`;
- inspect the keys.

If parsing fails, the prompt or model output needs attention. This is a real methodological issue: a model can produce plausible text that is not valid machine-readable JSON.


In [ ]:
# Answer 14
extraction_prompt = make_extraction_prompt(deidentified_service_note)
extraction_result = call_model(
    prompt=extraction_prompt,
    temperature=None,
    top_p=None,
    max_output_tokens=180,
)

print(f"REMOTE API CALLED: YES | provider={extraction_result['provider']} | task=structured_extraction | model={extraction_result['model']} | status={extraction_result['status']} | elapsed_seconds={extraction_result['elapsed_seconds']:.2f}")
extracted_fields = json.loads(extraction_result["text"])
print(json.dumps(extracted_fields, indent=2))


### Question 15: Minimise and Redact Before Sending Text

Before sending text to a hosted model, ask whether each detail is needed.

Programming steps:

- redact obvious identifiers from the synthetic note;
- build a classification prompt from the redacted note;
- call the selected provider;
- print the redacted note and model label.

This is not a complete anonymisation procedure. It is a programming demonstration of data minimisation before a remote API call.


In [ ]:
# Answer 15
redacted_note = redact_for_api(sensitive_training_example)
redacted_prompt = make_label_prompt(redacted_note)

redacted_result = call_model(
    prompt=redacted_prompt,
    temperature=None,
    top_p=None,
    max_output_tokens=30,
)

print(f"REMOTE API CALLED: YES | provider={redacted_result['provider']} | task=redacted_note_classification | model={redacted_result['model']} | status={redacted_result['status']} | elapsed_seconds={redacted_result['elapsed_seconds']:.2f}")
print("Redacted note:")
print(redacted_note.strip())
print("\nModel text:", redacted_result["text"])
print("Model label:", normalise_label(redacted_result["text"]))


### Question 16: Final Mini-Case

Finish by documenting one responsible remote-model workflow.

Your report should include:

- the task;
- the provider and model;
- privacy step;
- parameters;
- validation evidence;
- likely failure modes.


In [ ]:
# Answer 16
mini_case = {
    "task": "classify short public comments",
    "provider_and_model": f"{api_result['provider']} / {api_result['model']}",
    "privacy_step": "use synthetic or approved de-identified text; redact direct identifiers before sending",
    "parameters": {
        "temperature": None,
        "top_p": None,
        "max_output_tokens": 30,
    },
    "validation": "compare model labels with human labels and inspect disagreements",
    "evidence_from_lab": {
        "single_note_validation": validation_result,
        "batch_rows": batch_rows,
        "classifier_evaluation": classifier_evaluation,
        "classifier_accuracy": classifier_evaluation["accuracy"],
    },
    "possible_failure_modes": [
        "wrong or unavailable provider/model name",
        "missing API key or model environment variable",
        "quota, billing, or project permission problem",
        "unsupported parameter for the selected provider/model",
        "invalid JSON in the extraction task",
        "model returns a label outside the allowed set",
        "sensitive data included before redaction",
    ],
}

print(json.dumps(mini_case, indent=2))


## End of Lab Checklist

By the end of this lab, you should have:

- chosen OpenAI or Gemini for Lab Three;
- checked the selected provider key and model environment variables;
- understood what each setup helper function does;
- built a provider-specific request payload;
- inspected a redacted request before sending it;
- made real remote API calls;
- printed explicit remote API call confirmation lines;
- parsed real response text;
- compared two parameter settings;
- created a reproducibility log entry;
- interpreted common API status codes;
- compared output with human labels;
- evaluated batch-classifier accuracy and disagreements;
- printed a single-line classifier accuracy summary;
- batch-classified short health and social notes;
- summarised a public feedback extract;
- extracted structured fields from a de-identified note;
- redacted obvious identifiers before sending text;
- described failure modes for a small applied workflow.
